# 05. Avaliar atribuição — mãe + endereço

Depois do [`04_atribuir_mae_endereco.ipynb`](04_atribuir_mae_endereco.ipynb).
Sem escada: avalia a nota dos atribuídos, a qualidade dos pares e o que
aconteceria com piso mais baixo (mesma regra: melhor nota + teto 3).

Entradas:
- `splink_predictions_mae_endereco.parquet` ([`02d`](02d_aplicar_splink_mae_endereco.ipynb))
- `splink_atribuicao_mae_endereco.parquet` ([`04`](04_atribuir_mae_endereco.ipynb))


In [ ]:
import sys
from pathlib import Path

from IPython.display import display

PROB_DIR = Path.cwd()
if PROB_DIR.name == 'notebooks':
    PROB_DIR = PROB_DIR.parent
if str(PROB_DIR) not in sys.path:
    sys.path.insert(0, str(PROB_DIR))

from config import (
    LISTA_OURO_ARQUIVO,
    MAX_CENSOS_POR_CPF,
    SPLINK_ATRIBUICAO_MAE_ENDERECO,
    SPLINK_INPUT_VIEW,
    SPLINK_PREDICTIONS_MAE_ENDERECO,
    TABELA_CENSO_LIMPA,
    TABELA_CENSO_LIMPA_APLICACAO,
    TABELA_CPF_LIMPA_APLICACAO,
    cpf_norm_sql,
    drop_splink_temp_tables,
    get_connection,
    lista_ouro_colunas,
    materialize_splink_input,
    print_paths,
    require_input,
    require_tables,
)

PISO = 0.50
TETO = MAX_CENSOS_POR_CPF
PISOS_CONTRAFACTUAL = (0.50, 0.25, 0.10, 0.05)

print_paths()
require_input(
    SPLINK_PREDICTIONS_MAE_ENDERECO,
    label='SPLINK_PREDICTIONS_MAE_ENDERECO (rode o 02d antes)',
)
require_input(
    SPLINK_ATRIBUICAO_MAE_ENDERECO,
    label='SPLINK_ATRIBUICAO_MAE_ENDERECO (rode o 04_atribuir_mae_endereco antes)',
)
require_input(LISTA_OURO_ARQUIVO, label='LISTA_OURO')
con = get_connection()
drop_splink_temp_tables(con)
require_tables(
    con,
    [TABELA_CENSO_LIMPA, TABELA_CENSO_LIMPA_APLICACAO, TABELA_CPF_LIMPA_APLICACAO],
    notebook_origem='00b',
)
materialize_splink_input(
    con,
    censo_table=TABELA_CENSO_LIMPA_APLICACAO,
    cpf_table=TABELA_CPF_LIMPA_APLICACAO,
)
print('Piso atual:', PISO, '| teto:', TETO)


In [ ]:
_tipo = con.execute('''
SELECT table_type
FROM information_schema.tables
WHERE table_schema = 'main' AND table_name = 'splink_predictions'
''').fetchone()
if _tipo:
    _kind = 'VIEW' if _tipo[0].upper() == 'VIEW' else 'TABLE'
    con.execute(f'DROP {_kind} IF EXISTS splink_predictions')

con.execute(f'''
CREATE OR REPLACE VIEW splink_predictions AS
SELECT
    CASE
        WHEN unique_id_l LIKE 'censo_%' THEN unique_id_l
        ELSE unique_id_r
    END AS unique_id_censo,
    CASE
        WHEN unique_id_l LIKE 'censo_%' THEN unique_id_r
        ELSE unique_id_l
    END AS unique_id_cpf,
    match_probability
FROM read_parquet('{SPLINK_PREDICTIONS_MAE_ENDERECO}')
''')

con.execute(f'''
CREATE OR REPLACE VIEW atribuicao AS
SELECT unique_id_censo, unique_id_cpf, match_probability
FROM read_parquet('{SPLINK_ATRIBUICAO_MAE_ENDERECO}')
''')

n_app = con.execute(f'''
SELECT COUNT(*) FROM {SPLINK_INPUT_VIEW} WHERE origem = 'censo'
''').fetchone()[0]
n_atrib = con.execute('SELECT COUNT(*) FROM atribuicao').fetchone()[0]
n_pred = con.execute('SELECT COUNT(*) FROM splink_predictions').fetchone()[0]

print(f'Aplicação: {n_app:,}')
print(f'Atribuídos: {n_atrib:,} ({100.0 * n_atrib / n_app:.1f}%)')
print(f'Sem CPF: {n_app - n_atrib:,} ({100.0 * (n_app - n_atrib) / n_app:.1f}%)')
print(f'Pares no parquet: {n_pred:,}')


## Notas dos atribuídos

Faixas de `match_probability` e quantos Censos cada CPF levou (máx. 3).


In [ ]:
display(con.execute('''
SELECT
    CASE
        WHEN match_probability >= 0.99 THEN 'p_ge_099'
        WHEN match_probability >= 0.90 THEN 'p_090_099'
        WHEN match_probability >= 0.75 THEN 'p_075_090'
        WHEN match_probability >= 0.50 THEN 'p_050_075'
        ELSE 'p_lt_050'
    END AS faixa,
    COUNT(*) AS n,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct
FROM atribuicao
GROUP BY 1
ORDER BY 1 DESC
''').df())

display(con.execute('''
SELECT
    n_censos,
    COUNT(*) AS n_cpf,
    n_censos * COUNT(*) AS n_pares
FROM (
    SELECT unique_id_cpf, COUNT(*) AS n_censos
    FROM atribuicao
    GROUP BY 1
)
GROUP BY 1
ORDER BY 1
''').df())


## Qualidade dos pares atribuídos

No par que saiu: mãe, endereço e distância de ano de nascimento.


In [ ]:
display(con.execute(f'''
SELECT
    COUNT(*) AS n,
    ROUND(100.0 * COUNT(*) FILTER (
        WHERE ca.nome_mae_phon IS NOT NULL AND pb.nome_mae_phon IS NOT NULL
    ) / COUNT(*), 1) AS pct_mae_preenchida,
    ROUND(100.0 * COUNT(*) FILTER (
        WHERE ca.nome_mae_phon IS NOT NULL AND pb.nome_mae_phon IS NOT NULL
          AND ca.nome_mae_phon = pb.nome_mae_phon
    ) / COUNT(*), 1) AS pct_mae_igual,
    ROUND(100.0 * COUNT(*) FILTER (
        WHERE ca.logradouro_norm IS NOT NULL AND pb.logradouro_norm IS NOT NULL
          AND ca.logradouro_norm = pb.logradouro_norm
    ) / COUNT(*), 1) AS pct_logradouro_igual,
    ROUND(100.0 * COUNT(*) FILTER (
        WHERE ca.cep IS NOT NULL AND pb.cep IS NOT NULL AND ca.cep = pb.cep
    ) / COUNT(*), 1) AS pct_cep_igual,
    ROUND(100.0 * COUNT(*) FILTER (
        WHERE ca.data_nascimento IS NOT NULL AND pb.data_nascimento IS NOT NULL
          AND ca.data_nascimento = pb.data_nascimento
    ) / COUNT(*), 1) AS pct_data_igual
FROM atribuicao a
JOIN {SPLINK_INPUT_VIEW} ca ON ca.unique_id = a.unique_id_censo
JOIN {SPLINK_INPUT_VIEW} pb ON pb.unique_id = a.unique_id_cpf
''').df())

display(con.execute(f'''
SELECT
    CASE
        WHEN ca.data_nascimento IS NULL OR pb.data_nascimento IS NULL THEN 'sem_data'
        WHEN abs(
            year(CAST(ca.data_nascimento AS DATE))
            - year(CAST(pb.data_nascimento AS DATE))
        ) = 0 THEN 'delta_0'
        WHEN abs(
            year(CAST(ca.data_nascimento AS DATE))
            - year(CAST(pb.data_nascimento AS DATE))
        ) <= 1 THEN 'delta_1'
        WHEN abs(
            year(CAST(ca.data_nascimento AS DATE))
            - year(CAST(pb.data_nascimento AS DATE))
        ) <= 5 THEN 'delta_2_5'
        WHEN abs(
            year(CAST(ca.data_nascimento AS DATE))
            - year(CAST(pb.data_nascimento AS DATE))
        ) <= 10 THEN 'delta_6_10'
        ELSE 'delta_gt_10'
    END AS faixa_delta_ano,
    COUNT(*) AS n,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct
FROM atribuicao a
JOIN {SPLINK_INPUT_VIEW} ca ON ca.unique_id = a.unique_id_censo
JOIN {SPLINK_INPUT_VIEW} pb ON pb.unique_id = a.unique_id_cpf
GROUP BY 1
ORDER BY 1
''').df())


## Quem ficou de fora

Melhor par no parquet por Censo sem atribuição. Quem tinha `p ≥ 0,50` e
mesmo assim não saiu: caiu no teto (CPF com mais de 3 Censos no melhor-par,
mesmo `ORDER BY` do 04).


In [ ]:
# Mesmo desempate do 04: nota → mãe preenchida → logradouro → CEP.
con.execute(f'''
CREATE OR REPLACE TABLE melhor_qualquer AS
SELECT
    p.unique_id_censo,
    p.unique_id_cpf,
    p.match_probability
FROM splink_predictions p
JOIN {SPLINK_INPUT_VIEW} ca ON ca.unique_id = p.unique_id_censo
JOIN {SPLINK_INPUT_VIEW} pb ON pb.unique_id = p.unique_id_cpf
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY p.unique_id_censo
    ORDER BY
        p.match_probability DESC,
        CAST(
            ca.nome_mae_phon IS NOT NULL AND pb.nome_mae_phon IS NOT NULL
            AS INTEGER
        ) DESC,
        CAST(
            ca.logradouro_norm IS NOT NULL AND pb.logradouro_norm IS NOT NULL
            AND ca.logradouro_norm = pb.logradouro_norm
            AS INTEGER
        ) DESC,
        CAST(
            ca.cep IS NOT NULL AND pb.cep IS NOT NULL AND ca.cep = pb.cep
            AS INTEGER
        ) DESC,
        p.unique_id_cpf
) = 1
''')

con.execute(f'''
CREATE OR REPLACE TABLE melhor_piso AS
SELECT *
FROM melhor_qualquer
WHERE match_probability >= {PISO}
''')

con.execute(f'''
CREATE OR REPLACE TABLE censo_sem_cpf AS
SELECT c.unique_id AS unique_id_censo
FROM {SPLINK_INPUT_VIEW} c
WHERE c.origem = 'censo'
  AND NOT EXISTS (
    SELECT 1 FROM atribuicao a WHERE a.unique_id_censo = c.unique_id
  )
''')

con.execute(f'''
CREATE OR REPLACE TABLE sem_cpf_melhor AS
SELECT
    s.unique_id_censo,
    m.unique_id_cpf,
    m.match_probability,
    CASE
        WHEN m.match_probability IS NULL THEN 'sem_par'
        WHEN m.match_probability >= 0.50 THEN 'p_ge_050'
        WHEN m.match_probability >= 0.25 THEN 'p_025_050'
        WHEN m.match_probability >= 0.05 THEN 'p_005_025'
        ELSE 'p_lt_005'
    END AS faixa_p
FROM censo_sem_cpf s
LEFT JOIN melhor_qualquer m ON m.unique_id_censo = s.unique_id_censo
''')

n_sem = con.execute('SELECT COUNT(*) FROM censo_sem_cpf').fetchone()[0]
assert n_atrib + n_sem == n_app

display(con.execute(f'''
SELECT
    a.faixa_p,
    coalesce(g.n, 0) AS n,
    ROUND(100.0 * coalesce(g.n, 0) / {n_sem}, 1) AS pct_sem_cpf,
    ROUND(100.0 * coalesce(g.n, 0) / {n_app}, 1) AS pct_aplicacao
FROM (VALUES
    (1, 'p_ge_050'),
    (2, 'p_025_050'),
    (3, 'p_005_025'),
    (4, 'p_lt_005'),
    (5, 'sem_par')
) a(ord, faixa_p)
LEFT JOIN (
    SELECT faixa_p, COUNT(*) AS n
    FROM sem_cpf_melhor
    GROUP BY 1
) g ON g.faixa_p = a.faixa_p
ORDER BY a.ord
''').df())

# Censos com p >= PISO no melhor-par, mas CPF estourou o teto.
con.execute(f'''
CREATE OR REPLACE TABLE cpf_n_melhor_piso AS
SELECT unique_id_cpf, COUNT(*) AS n_censos
FROM melhor_piso
GROUP BY 1
''')

n_teto = con.execute(f'''
SELECT COUNT(*)
FROM sem_cpf_melhor s
JOIN cpf_n_melhor_piso c ON c.unique_id_cpf = s.unique_id_cpf
WHERE s.match_probability >= {PISO}
  AND c.n_censos > {TETO}
''').fetchone()[0]
n_alto = con.execute(f'''
SELECT COUNT(*) FROM sem_cpf_melhor
WHERE match_probability >= {PISO}
''').fetchone()[0]
print(f'Sem CPF com melhor p >= {PISO}: {n_alto:,}')
print(f'  dos quais teto (CPF com >{TETO} no melhor-par): {n_teto:,}')


## Contrafactual de piso

Mesma regra do 04 (melhor nota + teto 3), só muda o piso.
Não altera o parquet exportado — só mostra cobertura se aceitasse notas
mais baixas.


In [ ]:
linhas = []
for piso_cand in PISOS_CONTRAFACTUAL:
    con.execute(f'''
    CREATE OR REPLACE TEMP TABLE melhor_cand AS
    SELECT
        p.unique_id_censo,
        p.unique_id_cpf,
        p.match_probability
    FROM splink_predictions p
    JOIN {SPLINK_INPUT_VIEW} ca ON ca.unique_id = p.unique_id_censo
    JOIN {SPLINK_INPUT_VIEW} pb ON pb.unique_id = p.unique_id_cpf
    WHERE p.match_probability >= {piso_cand}
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY p.unique_id_censo
        ORDER BY
            p.match_probability DESC,
            CAST(
                ca.nome_mae_phon IS NOT NULL AND pb.nome_mae_phon IS NOT NULL
                AS INTEGER
            ) DESC,
            CAST(
                ca.logradouro_norm IS NOT NULL AND pb.logradouro_norm IS NOT NULL
                AND ca.logradouro_norm = pb.logradouro_norm
                AS INTEGER
            ) DESC,
            CAST(
                ca.cep IS NOT NULL AND pb.cep IS NOT NULL AND ca.cep = pb.cep
                AS INTEGER
            ) DESC,
            p.unique_id_cpf
    ) = 1
    ''')
    n_melhor = con.execute('SELECT COUNT(*) FROM melhor_cand').fetchone()[0]
    n_ok = con.execute(f'''
    SELECT COUNT(*)
    FROM melhor_cand m
    JOIN (
        SELECT unique_id_cpf
        FROM melhor_cand
        GROUP BY 1
        HAVING COUNT(*) <= {TETO}
    ) c ON c.unique_id_cpf = m.unique_id_cpf
    ''').fetchone()[0]
    linhas.append({
        'piso': piso_cand,
        'n_melhor': n_melhor,
        'n_atribuidos': n_ok,
        'n_cortados_teto': n_melhor - n_ok,
        'cobertura_pct': round(100.0 * n_ok / n_app, 1),
    })

import pandas as pd

contra = pd.DataFrame(linhas)
base = contra.loc[contra['piso'] == PISO, 'n_atribuidos'].iloc[0]
contra['ganho_vs_050'] = contra['n_atribuidos'] - base
display(contra)


## Lista ouro (censo_limpo)

Universo com a lista ainda dentro. Quem está na ouro e no pareamento conta
como `lista_ouro`. `nos_dois` tem de ser zero (essa gente saiu da aplicação
antes do 04).


In [ ]:
ouro_cols = {
    r[0] for r in con.execute(
        f"DESCRIBE SELECT * FROM read_parquet('{LISTA_OURO_ARQUIVO}') LIMIT 0"
    ).fetchall()
}
id_ouro, cpf_ouro = lista_ouro_colunas(ouro_cols)

con.execute(f'''
CREATE OR REPLACE TABLE lista_ouro AS
SELECT
    CAST({id_ouro} AS VARCHAR) AS id_censo,
    {cpf_norm_sql(cpf_ouro)} AS cpf_norm
FROM read_parquet('{LISTA_OURO_ARQUIVO}')
WHERE {id_ouro} IS NOT NULL AND {cpf_ouro} IS NOT NULL
''')

display(con.execute(f'''
SELECT
    a.origem,
    coalesce(g.n, 0) AS n,
    ROUND(100.0 * coalesce(g.n, 0) / nullif(t.n_censo, 0), 1) AS pct
FROM (VALUES
    (1, 'lista_ouro'),
    (2, 'pareamento'),
    (3, 'sem_cpf')
) a(ord, origem)
CROSS JOIN (
    SELECT COUNT(*) AS n_censo FROM {TABELA_CENSO_LIMPA}
) t
LEFT JOIN (
    SELECT
        CASE
            WHEN o.id_censo IS NOT NULL THEN 'lista_ouro'
            WHEN p.unique_id_censo IS NOT NULL THEN 'pareamento'
            ELSE 'sem_cpf'
        END AS origem,
        COUNT(*) AS n
    FROM {TABELA_CENSO_LIMPA} c
    LEFT JOIN (
        SELECT DISTINCT id_censo FROM lista_ouro
    ) o ON o.id_censo = CAST(c.person_id_censo AS VARCHAR)
    LEFT JOIN (
        SELECT DISTINCT unique_id_censo FROM atribuicao
    ) p ON p.unique_id_censo = c.unique_id
    GROUP BY 1
) g ON g.origem = a.origem
ORDER BY a.ord
''').df())

n_ambos = con.execute(f'''
SELECT COUNT(*)
FROM {TABELA_CENSO_LIMPA} c
JOIN (SELECT DISTINCT id_censo FROM lista_ouro) o
  ON o.id_censo = CAST(c.person_id_censo AS VARCHAR)
JOIN (SELECT DISTINCT unique_id_censo FROM atribuicao) p
  ON p.unique_id_censo = c.unique_id
''').fetchone()[0]
print('nos_dois:', f'{n_ambos:,}')
assert n_ambos == 0


In [ ]:
con.close()
print('Fim do 05.')
